In [ ]:
import math
import random
from random import randint
import tiktoken
import torch
import torch.nn as nn
from torch.nn import functional as F
from dataclasses import dataclass
num_return_sequences=1
max_length=1024
Total_number_batch=524288
micro_number_batch=4
assert Total_number_batch%(micro_number_batch*max_length)==0
grad_accum_steps=Total_number_batch//(micro_number_batch*max_length)
device='cuda'
enc=tiktoken.get_encoding('gpt2')

In [ ]:
text=open("shakespeare.txt",'r').read()
data=enc.encode(text)
n=int(len(data)*0.9)
train_data=data[:n]
val_data=data[n:]

In [ ]:
random.seed(1337)
warmup_steps=10
max_steps=50
max_lr=6e-4
min_lr=max_lr*0.1
def get_lr(it):
    if it<warmup_steps:
        return max_lr*(it+1)/warmup_steps
    if it>max_steps:
        return min_lr
    decay_ratio=(it-warmup_steps)/(max_steps-warmup_steps)
    assert 0<= decay_ratio<=1
    coeff=0.5*(1.0+math.cos(math.pi*decay_ratio))
    return min_lr+coeff*(max_lr-min_lr)
def get_Batch(data):
    in_batch=[]
    out_batch=[]
    for i in range (micro_number_batch):
        debut=randint(0,len(data)-max_length)
        in_batch+=data[debut:max_length+debut]
        out_batch+=data[1+debut:max_length+1+debut]
    in_batch=torch.tensor(in_batch)
    out_batch=torch.tensor(out_batch)
    return (in_batch.view(micro_number_batch,-1),out_batch.view(micro_number_batch,-1))

In [ ]:
torch.manual_seed(1337)
@dataclass
class GPTConfig:
    block_size: int = max_length
    vocab_size: int = 50304
    n_layer: int = 12
    n_head: int = 12
    n_embd: int = 768

class CausalSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd)
        self.c_proj = nn.Linear(config.n_embd, config.n_embd)
        self.c_proj.GPT_SCALE_INIT=1
        self.n_head = config.n_head
        self.n_embd = config.n_embd
        self.register_buffer("bias", torch.tril(torch.ones(config.block_size, config.block_size))
                                    .view(1, 1, config.block_size, config.block_size))
    def forward(self, x):
        B, T, C = x.size() 
        q, k, v  = self.c_attn(x).split(self.n_embd, dim=2)
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) 
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) 
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) 
        #att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(k.size(-1)))
        #att = att.masked_fill(self.bias[:,:,:T,:T] == 0, float('-inf'))
        #att = F.softmax(att, dim=-1)       
        #y = att @ v
        y=F.scaled_dot_product_attention(q,k,v,is_causal=True)
        y = y.transpose(1, 2).contiguous().view(B, T, C) 
        return self.c_proj(y)
class MLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.c_fc = nn.Linear(config.n_embd, 4 * config.n_embd)
        self.gelu = nn.GELU(approximate='tanh') 
        self.c_proj = nn.Linear(4 * config.n_embd, config.n_embd) 
        self.c_proj.GPT_SCALE_INIT=1
    def forward(self, x):
        x = self.c_fc(x)
        x = self.gelu(x)
        x = self.c_proj(x)
        return x 

class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.ln_1 = nn.LayerNorm(config.n_embd)
        self.attn = CausalSelfAttention(config)
        self.ln_2 = nn.LayerNorm(config.n_embd)        
        self.mlp = MLP(config)

    def forward(self, x):
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x

class GPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config        
        self.transformer = nn.ModuleDict(dict(
            wte = nn.Embedding(config.vocab_size, config.n_embd),
            wpe = nn.Embedding(config.block_size, config.n_embd),
            h = nn.ModuleList([Block(config) for _ in range(config.n_layer)]),
            ln_f = nn.LayerNorm(config.n_embd),
        ))
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        self.transformer.wte.weight=self.lm_head.weight
        self.apply(self.__init__weights)
    def __init__weights(self, module):
        if isinstance(module,nn.Linear):
            std=0.02
            if hasattr(module,"GPT_SCALE_INIT"):
                std*=(2*self.config.n_layer)**-0.5
            torch.nn.init.normal_(module.weight,mean=0.0,std=std)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module,nn.Embedding):
            torch.nn.init.normal_(module.weight,mean=0.0,std=0.02)
    def forward(self,idx,targets=None):
        B,T=idx.shape
        token_embd=self.transformer.wte(idx)
        pos_embd=self.transformer.wpe(torch.arange(T,device=device))
        x=token_embd+pos_embd
        for blocks in self.transformer.h:
            x=blocks(x)
        x=self.transformer.ln_f(x)
        logits=self.lm_head(x)
        if targets==None:
            loss=None
        else:
            A,B,C=logits.shape
            logits=logits.view(A*B,C)
            targets=targets.view(A*B)
            loss=F.cross_entropy(logits,targets)
        return logits,loss
    torch.manual_seed(42)
    torch.cuda.manual_seed(42)
    def generate(self,idx,max_new_tokens):
        for i in range (max_new_tokens):
            idx_cond=idx[:,-self.config.block_size:]
            logits,loss=self(idx_cond)
            logits=logits[:,-1,:]
            probs=F.softmax(logits,dim=-1)
            topk_probs,topk_indices=torch.topk(probs,50,dim=-1)
            ix =torch.multinomial(topk_probs, num_samples=1)
            idx_next = torch.gather(topk_indices,-1,ix)
            idx=torch.cat((idx,idx_next),dim=1)
        return idx
    @classmethod
    def from_pretrained(cls, model_type):
        """Loads pretrained GPT-2 model weights from huggingface"""
        assert model_type in {'gpt2', 'gpt2-medium', 'gpt2-large', 'gpt2-xl'}
        from transformers import GPT2LMHeadModel
        print("loading weights from pretrained gpt: %s" % model_type)

        config_args = {
            'gpt2':         dict(n_layer=12, n_head=12, n_embd=768),  
            'gpt2-medium':  dict(n_layer=24, n_head=16, n_embd=1024), 
            'gpt2-large':   dict(n_layer=36, n_head=20, n_embd=1280), 
            'gpt2-xl':      dict(n_layer=48, n_head=25, n_embd=1600), 
        }[model_type]
        config_args['vocab_size'] = 50257 
        config_args['block_size'] = 1024 
        
        config = GPTConfig(**config_args)
        model = GPT(config)
        sd = model.state_dict()
        sd_keys = sd.keys()
        sd_keys = [k for k in sd_keys if not k.endswith('.attn.bias')] 

        model_hf = GPT2LMHeadModel.from_pretrained(model_type)
        sd_hf = model_hf.state_dict()

        sd_keys_hf = sd_hf.keys()
        sd_keys_hf = [k for k in sd_keys_hf if not k.endswith('.attn.masked_bias')] 
        sd_keys_hf = [k for k in sd_keys_hf if not k.endswith('.attn.bias')] 
        transposed = ['attn.c_attn.weight', 'attn.c_proj.weight', 'mlp.c_fc.weight', 'mlp.c_proj.weight']
        
        assert len(sd_keys_hf) == len(sd_keys), f"mismatched keys: {len(sd_keys_hf)} != {len(sd_keys)}"
        for k in sd_keys_hf:
            if any(k.endswith(w) for w in transposed):
                assert sd_hf[k].shape[::-1] == sd[k].shape
                with torch.no_grad():
                    sd[k].copy_(sd_hf[k].t())
            else:
                assert sd_hf[k].shape == sd[k].shape
                with torch.no_grad():
                    sd[k].copy_(sd_hf[k])

        return model
    

In [ ]:
model=GPT(GPTConfig())
model.eval()
model.to('cuda')
print()

In [ ]:
decay_params = [p for n, p in model.named_parameters() if p.requires_grad and p.dim() >= 2]
nodecay_params = [p for n, p in model.named_parameters() if p.requires_grad and p.dim() < 2]

optim_groups = [
    {'params': decay_params, 'weight_decay': 0.1},
    {'params': nodecay_params, 'weight_decay': 0.0}
]
optimiser = torch.optim.AdamW(optim_groups, lr=3e-4, betas=(0.9, 0.95),eps=1e-8)

In [ ]:
#trainig the model in here:
steps=50
scaler = torch.cuda.amp.GradScaler()
import time
for i in range(steps):
    loss_accum=0
    t0=time.time()
    optimiser.zero_grad()
    lr=get_lr(i)
    for param_group in optimiser.param_groups:
        param_group['lr']=lr
    for micro_batch in range(grad_accum_steps):
        x,y=get_Batch(train_data)
        x=x.to('cuda')
        y=y.to('cuda')
        with torch.amp.autocast(device_type='cuda', dtype=torch.float16):    
            logits,loss=model(x,y)
        loss=loss/grad_accum_steps
        scaler.scale(loss).backward()
        loss_accum+=loss.detach()
    scaler.unscale_(optimiser)
    norm=torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
    scaler.step(optimiser) 
    scaler.update()
    torch.cuda.synchronize()
    t1=time.time()
    dt=(t1-t0)
    print(("{}/time(ms) {} ms loss={} tok/s={} norm={:.4f},lr={}").format(i,int(dt*1000),loss_accum.item(),Total_number_batch/dt,norm,lr))

In [ ]:
request="oh lord"
tokens=enc.encode(request)
tokens=torch.tensor(tokens,dtype=torch.long)
tokens=tokens.unsqueeze(0).repeat(num_return_sequences,1)
x=tokens.to('cuda')
for i in range(num_return_sequences):
    tokens=model.generate(idx =x,max_new_tokens=50)[0].tolist()
    decoded=enc.decode(tokens)
    print("<>",decoded)